# UK Inflation vs Salary Growth Analysis | 2010 to 2025

**Data Sources:**
- ONS Consumer Price Index (CPI), series D7G7
- ONS Annual Survey of Hours and Earnings (ASHE), median earnings by sector

**Tools:** Python, Pandas, Matplotlib, NumPy


## Objective

Has your salary actually kept up with inflation over the last 15 years?

This notebook explores how UK wages across different sectors have kept up with inflation, with a particular focus on the 2022 cost of living crisis and its long-term impact on purchasing power.

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

## 2. Load Data

Data sourced from:
- **ONS CPI series D7G7** — annual inflation rate
- **ONS ASHE** — median annual wage growth by sector

In [ ]:
# Years covered
years = list(range(2010, 2026))

# CPI Annual Inflation Rate (ONS D7G7 series)
cpi = [3.3, 4.5, 2.8, 2.6, 1.5, 0.0, 0.7, 2.7, 2.5, 1.8, 0.9, 2.6, 9.1, 10.1, 3.2, 2.6]

# Annual Wage Growth by Sector (ONS ASHE, median % change)
wage_all     = [2.1, 1.8, 1.9, 2.2, 2.7, 1.7, 2.6, 3.1, 3.5, 3.6, 4.3, 4.6, 5.0, 7.8, 5.7, 4.1]
wage_finance = [2.5, 2.0, 2.3, 2.8, 3.1, 2.1, 3.0, 3.5, 4.2, 4.0, 3.8, 4.9, 5.3, 8.5, 6.2, 4.8]
wage_health  = [1.8, 1.5, 1.2, 1.1, 1.0, 1.0, 1.1, 2.5, 2.8, 3.2, 5.0, 3.5, 4.7, 6.4, 5.1, 3.8]
wage_tech    = [3.2, 3.5, 3.8, 4.1, 4.5, 4.0, 4.8, 5.2, 5.8, 5.5, 4.9, 5.8, 7.2, 9.1, 7.4, 5.9]
wage_retail  = [1.5, 1.2, 1.4, 1.9, 2.2, 1.5, 2.1, 2.8, 3.0, 3.1, 5.0, 4.2, 4.8, 6.8, 4.9, 3.5]

# Create summary DataFrame
df = pd.DataFrame({
    'Year': years,
    'CPI': cpi,
    'All Sectors': wage_all,
    'Finance': wage_finance,
    'Healthcare': wage_health,
    'Technology': wage_tech,
    'Retail': wage_retail
}).set_index('Year')

print("Data loaded successfully")
print(f"Years covered: {years[0]} to {years[-1]}")
print(f"Peak inflation year: {years[cpi.index(max(cpi))]} ({max(cpi)}%)")
df.head(10)

## 3. Calculate Real Wage Growth and Purchasing Power

**Real Wage Growth** = Wage Growth minus Inflation

**Cumulative Purchasing Power** = Rolling product of (1 + real_growth/100), base 100 in 2010

In [ ]:
def real_wage(wages, cpi):
    """Calculate real wage growth (wage growth minus inflation)"""
    return [round(w - c, 2) for w, c in zip(wages, cpi)]

def cumulative_power(real_wages, base=100):
    """Calculate cumulative purchasing power index"""
    power = [base]
    for r in real_wages[:-1]:
        power.append(round(power[-1] * (1 + r / 100), 2))
    return power

# Real wage growth by sector
real_all     = real_wage(wage_all, cpi)
real_finance = real_wage(wage_finance, cpi)
real_health  = real_wage(wage_health, cpi)
real_tech    = real_wage(wage_tech, cpi)
real_retail  = real_wage(wage_retail, cpi)

# Cumulative purchasing power
cum_all     = cumulative_power(real_all)
cum_finance = cumulative_power(real_finance)
cum_health  = cumulative_power(real_health)
cum_tech    = cumulative_power(real_tech)
cum_retail  = cumulative_power(real_retail)

# Summary stats
summary = pd.DataFrame({
    'Sector': ['Technology', 'Finance', 'All Sectors', 'Healthcare', 'Retail'],
    'Purchasing Power 2025': [cum_tech[-1], cum_finance[-1], cum_all[-1], cum_health[-1], cum_retail[-1]],
    'Years Beating Inflation': [
        sum(1 for r in real_tech if r > 0),
        sum(1 for r in real_finance if r > 0),
        sum(1 for r in real_all if r > 0),
        sum(1 for r in real_health if r > 0),
        sum(1 for r in real_retail if r > 0)
    ],
    'Avg Annual Real Growth (%)': [
        round(np.mean(real_tech), 2),
        round(np.mean(real_finance), 2),
        round(np.mean(real_all), 2),
        round(np.mean(real_health), 2),
        round(np.mean(real_retail), 2)
    ]
}).set_index('Sector')

print("Calculations complete")
summary

## 4. Visualisation 1 — UK CPI Inflation Rate (2010 to 2025)

In [ ]:
NAVY = '#1F3A6E'; BLUE = '#2563EB'; GREEN = '#16A34A'
RED = '#DC2626'; ORANGE = '#EA580C'; PURPLE = '#7C3AED'; GREY = '#6B7280'

fig, ax = plt.subplots(figsize=(12, 5))
colors = [RED if c > 5 else ORANGE if c > 3 else BLUE for c in cpi]
bars = ax.bar(years, cpi, color=colors, alpha=0.85, width=0.7)
ax.axhline(y=2, color=GREEN, linestyle='--', linewidth=1.5, label='Bank of England Target (2%)')
ax.axhline(y=0, color=GREY, linestyle='-', linewidth=0.5)
ax.set_title('UK CPI Inflation Rate | 2010 to 2025', fontsize=14, fontweight='bold', color=NAVY, pad=12)
ax.set_ylabel('Annual % Change')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar, val in zip(bars, cpi):
    if val > 5:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val}%', ha='center', va='bottom', fontsize=9, fontweight='bold', color=RED)

plt.xticks(years, rotation=45)
plt.tight_layout()
plt.show()
print(f"Peak inflation: {max(cpi)}% in {years[cpi.index(max(cpi))]}")
print(f"Years above Bank of England 2% target: {sum(1 for c in cpi if c > 2)} out of {len(cpi)}")

## 5. Visualisation 2 — Wage Growth vs CPI Inflation by Sector

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(years, cpi, color=RED, linewidth=2.5, label='CPI Inflation', linestyle='--', marker='o', markersize=5)
ax.plot(years, wage_all, color=NAVY, linewidth=2.5, label='All Sectors', marker='o', markersize=5)
ax.plot(years, wage_tech, color=BLUE, linewidth=2, label='Technology', marker='s', markersize=5)
ax.plot(years, wage_finance, color=GREEN, linewidth=2, label='Finance', marker='^', markersize=5)
ax.fill_between(years, cpi, wage_all,
                where=[w > c for w, c in zip(wage_all, cpi)], alpha=0.15, color=GREEN, label='Real gain')
ax.fill_between(years, cpi, wage_all,
                where=[w < c for w, c in zip(wage_all, cpi)], alpha=0.15, color=RED, label='Real loss')
ax.set_title('Wage Growth vs CPI Inflation | 2010 to 2025', fontsize=14, fontweight='bold', color=NAVY, pad=12)
ax.set_ylabel('Annual % Change')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(years, rotation=45)
plt.tight_layout()
plt.show()

## 6. Visualisation 3 — Real Wage Growth by Sector

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(years, real_all,     color=NAVY,   linewidth=2.5, label='All Sectors',  marker='o', markersize=5)
ax.plot(years, real_tech,    color=BLUE,   linewidth=2,   label='Technology',   marker='s', markersize=5)
ax.plot(years, real_finance, color=GREEN,  linewidth=2,   label='Finance',      marker='^', markersize=5)
ax.plot(years, real_health,  color=ORANGE, linewidth=2,   label='Healthcare',   marker='D', markersize=5)
ax.plot(years, real_retail,  color=PURPLE, linewidth=2,   label='Retail',       marker='v', markersize=5)
ax.axhline(y=0, color=GREY, linestyle='--', linewidth=1.5)
ax.fill_between(years, 0, real_tech, where=[r > 0 for r in real_tech], alpha=0.08, color=BLUE)
ax.set_title('Real Wage Growth by Sector | 2010 to 2025
(Wage Growth minus Inflation)', fontsize=14, fontweight='bold', color=NAVY, pad=12)
ax.set_ylabel('Real % Change')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(years, rotation=45)
plt.tight_layout()
plt.show()

## 7. Visualisation 4 — Cumulative Purchasing Power (2010 = 100)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(years, cum_tech,    color=BLUE,   linewidth=2.5, label=f'Technology  (2025: {cum_tech[-1]:.1f})',   marker='s', markersize=5)
ax.plot(years, cum_finance, color=GREEN,  linewidth=2.5, label=f'Finance     (2025: {cum_finance[-1]:.1f})', marker='^', markersize=5)
ax.plot(years, cum_all,     color=NAVY,   linewidth=2.5, label=f'All Sectors (2025: {cum_all[-1]:.1f})',    marker='o', markersize=5)
ax.plot(years, cum_health,  color=ORANGE, linewidth=2,   label=f'Healthcare  (2025: {cum_health[-1]:.1f})', marker='D', markersize=5)
ax.plot(years, cum_retail,  color=PURPLE, linewidth=2,   label=f'Retail      (2025: {cum_retail[-1]:.1f})', marker='v', markersize=5)
ax.axhline(y=100, color=RED, linestyle='--', linewidth=1.5, label='2010 Baseline (100)')
ax.fill_between(years, 100, cum_tech, where=[c > 100 for c in cum_tech], alpha=0.08, color=BLUE)
ax.fill_between(years, 100, cum_all,  where=[c < 100 for c in cum_all],  alpha=0.08, color=RED)

ax.annotate('Cost of Living Crisis Peak\nOct 2022: 11.1% CPI',
            xy=(2022, cum_all[12]), xytext=(2019.5, 90),
            fontsize=9, color=RED, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

ax.set_title('Cumulative Real Purchasing Power by Sector | Base 100 in 2010\nHas your salary kept up with inflation?',
             fontsize=14, fontweight='bold', color=NAVY, pad=12)
ax.set_ylabel('Purchasing Power Index (2010 = 100)')
ax.legend(fontsize=9, loc='upper left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(years, rotation=45)
plt.tight_layout()
plt.show()

## 8. Key Findings

In [ ]:
print("=" * 60)
print("KEY FINDINGS: UK INFLATION vs SALARY ANALYSIS")
print("=" * 60)
print()
print(f"Peak inflation year: {years[cpi.index(max(cpi))]} — CPI at {max(cpi)}%")
print()
print("Purchasing Power in 2025 (Base 100 in 2010):")
print(f"  Technology:  {cum_tech[-1]:.1f}  (+{cum_tech[-1]-100:.1f}%) WINNER")
print(f"  Finance:     {cum_finance[-1]:.1f}  (+{cum_finance[-1]-100:.1f}%)")
print(f"  All Sectors: {cum_all[-1]:.1f}  (+{cum_all[-1]-100:.1f}%)")
print(f"  Healthcare:  {cum_health[-1]:.1f}  ({cum_health[-1]-100:.1f}%) BELOW BASELINE")
print(f"  Retail:      {cum_retail[-1]:.1f}  ({cum_retail[-1]-100:.1f}%) BELOW BASELINE")
print()
print("Years wages beat inflation (out of 15):")
for sector, real in [("Technology", real_tech), ("Finance", real_finance),
                      ("All Sectors", real_all), ("Healthcare", real_health), ("Retail", real_retail)]:
    wins = sum(1 for r in real if r > 0)
    print(f"  {sector}: {wins}/15 years")
print()
print("Worst year for real wages: 2022")
print(f"  All sectors real growth: {real_all[12]}%")
print(f"  Healthcare real growth:  {real_health[12]}%")
print()
print("Recovery: 2023-2024 saw strong nominal wage growth")
print("but full purchasing power recovery remains incomplete for most sectors.")

## 9. Save Full Dashboard

In [ ]:
# Run this cell to save the full 6-chart dashboard as PNG
import subprocess
result = subprocess.run(['python3', 'UK_Inflation_vs_Salary_Analysis.py'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)

## 10. Conclusion

This analysis reveals a clear divergence in how different UK sectors have protected workers from inflation over the last 15 years:

**Winners:**
- **Technology** consistently outpaced inflation, delivering approximately +34% real purchasing power gain since 2010
- **Finance** also outperformed, ending approximately +10% above 2010 baseline

**Losers:**
- **Healthcare** and **Retail** workers have seen purchasing power eroded despite nominal pay rises
- The 2022 cost of living crisis was a shock event that temporarily reversed years of progress for all sectors

**Implications:**
- Sector choice matters significantly to long-term financial wellbeing
- Nominal pay rises can be misleading without adjusting for inflation
- The Bank of England's 2% inflation target matters practically for workers, not just economists

